# L86 · 采样策略从零实现 — temperature / top-k / top-p，纯 NumPy

**学习目标**
- 理解为什么贪婪解码（Greedy Decoding）容易产生重复
- 实现 temperature scaling、top-k、top-p（nucleus）采样
- 直观对比三种策略的分布形状
- 与 aurora.llm.sample 对比验证

## 贪婪解码的问题

贪婪解码每步选概率最大的 token，确定性高但容易"卡死"在局部最优——
比如"the the the the..."——因为一旦选了"the"，下一步"the"可能仍然概率最大。

**解决方案**：在采样时引入随机性，让分布\'正确\'地宽/窄。

**Temperature T**：logits / T → T<1 更尖锐（贪婪），T>1 更平坦（随机）

**Top-k**：只保留概率最高的 k 个 token，从中采样。

**Top-p / Nucleus**（Holtzman 2020）：保留累积概率 ≥ p 的最小词集，从中采样。核自适应大小，置信高时小，置信低时大。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
def softmax(logits, temperature=1.0):
    """数值稳定的 softmax，带 temperature 缩放。"""
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits / max(float(temperature), 1e-8)
    logits -= logits.max()
    exp = np.exp(logits)
    return exp / exp.sum()

# 测试 logit 分布
VOCAB = 20
logits = np.random.randn(VOCAB) * 2   # 模拟尖锐分布

print('Temperature 对分布的影响：')
for T in [0.5, 1.0, 2.0]:
    probs = softmax(logits, T)
    entropy = -np.sum(probs * np.log(probs + 1e-30))
    print(f'  T={T}: max_prob={probs.max():.3f}, entropy={entropy:.2f}')

## ✏️ 任务 1：Top-k 采样

In [ ]:
def top_k_sample(logits, k=10, temperature=1.0):
    """从前 k 个 logit 最大的 token 中采样。"""
    logits = np.asarray(logits, dtype=np.float64)
    k = max(1, min(k, len(logits)))
    # ✏️ TODO: 找到 top-k 索引，其余设为 -inf，softmax，采样
    raise NotImplementedError("TODO")

## ✏️ 任务 2：Top-p（Nucleus）采样

In [ ]:
def top_p_sample(logits, p=0.9, temperature=1.0):
    """从累积概率 >= p 的最小 token 集中采样。"""
    logits = np.asarray(logits, dtype=np.float64)
    probs = softmax(logits, temperature)
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    # ✏️ TODO: 累积求和，找到第一个 >= p 的位置
    raise NotImplementedError("TODO")

In [ ]:
# 可视化四种策略的采样分布
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
probs_base = softmax(logits, 1.0)
axes = axes.flatten()

strategies = [
    ('Greedy', [np.argmax(logits)]),
    ('Temperature T=0.5', [top_k_sample(logits, k=VOCAB, temperature=0.5) for _ in range(500)]),
    ('Top-k k=5', [top_k_sample(logits, k=5) for _ in range(500)]),
    ('Top-p p=0.9', [top_p_sample(logits, p=0.9) for _ in range(500)]),
]

for ax, (name, samples) in zip(axes, strategies):
    counts = np.bincount(samples, minlength=VOCAB) / max(len(samples), 1)
    ax.bar(range(VOCAB), counts, alpha=0.7, label='sample freq')
    ax.plot(range(VOCAB), probs_base, 'r--', alpha=0.5, label='base prob')
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Token ID')
    ax.set_ylabel('频率')
    ax.legend(fontsize=8)

plt.suptitle('四种解码策略的 token 采样分布对比', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
try:
    # 测试 top_k_sample 返回值在 top-k 集合内
    test_logits = np.random.randn(50)
    k = 10
    result_topk = top_k_sample(test_logits, k=k)
    assert result_topk is not None, "⬜ 未实现"
    assert isinstance(result_topk, (int, np.integer)), f"应返回整数索引，得到 {type(result_topk)}"
    top_k_indices = set(np.argpartition(test_logits, -k)[-k:].tolist())
    assert result_topk in top_k_indices, f"返回值 {result_topk} 不在 top-{k} 集合内"
    print(f'✅ top_k_sample 返回: {result_topk}（在 top-{k} 集合内）')

    # 测试 top_p_sample 返回合法索引
    result_topp = top_p_sample(test_logits, p=0.9)
    assert isinstance(result_topp, (int, np.integer)), f"应返回整数索引，得到 {type(result_topp)}"
    assert 0 <= result_topp < len(test_logits), f"返回值 {result_topp} 超出范围"
    print(f'✅ top_p_sample 返回: {result_topp}（合法索引）')

    # 与 aurora.llm 对比
    from aurora.llm import top_k_sample as ref_topk, top_p_sample as ref_topp
    ref_result = ref_topk(test_logits, k=k)
    ref_top_k_indices = set(np.argpartition(test_logits, -k)[-k:].tolist())
    assert ref_result in ref_top_k_indices, f"aurora.llm 参考实现返回值 {ref_result} 不在 top-{k} 集合内"
    print('✅ 采样策略验证通过')
except NotImplementedError:
    print('⬜ 未实现')

## 本课收束

| 策略 | 参数 | 效果 |
|---|---|---|
| 贪婪 | 无 | 确定性，容易重复 |
| Temperature | T | T<1 更尖锐，T>1 更随机 |
| Top-k | k | 固定候选数量 |
| Top-p | p | 自适应候选集，置信高→小核 |

实际 LLM 通常组合使用：`top_p_sample(logits, p=0.9, temperature=0.8)`。

下一步 L87：INT8 量化原理 + 连接 HuggingFace 运行真实推理。